# SecureMed Chat (work in progress notebook)
# Our mission is to create a private medical chatbot to help you prepare for your doctor visit!

In [3]:
# ==============================================================================
# Step 1: Configuration & Setup for Google Vertex AI (ChromaDB Version)
# ==============================================================================
import os
import subprocess
import time
from pathlib import Path
os.environ["GRPC_DNS_RESOLVER"] = "native"

# --- Configuration Settings ---
KNOWLEDGE_BASE_DIR = Path("knowledge_base")
REPORTS_DIR = Path("reports")
# This is now for the ChromaDB persistence on the VM, not your local machine.
VECTOR_STORE_PATH = Path("vector_store_chroma")

# --- ChromaDB Client Configuration ---
# Replace with your VM's external IP address
CHROMA_HOST = "34.151.247.35" 
CHROMA_PORT = "8000"
COLLECTION_NAME = "securemed_docs" # Name for your collection in Chroma

# --- Google Cloud & Vertex AI Configuration ---
PROJECT_ID = "securemed-chat"
REGION = "us-central1"
EMBEDDING_MODEL = "gemini-embedding-001" # Updated to the latest embedding model
LLM_MODEL = "gemini-2.5-flash-lite"    # Updated to a powerful and cost-effective model

# --- Text Splitting & Batching (no changes) ---
CHUNK_SIZE = 750
CHUNK_OVERLAP = 100
BATCH_SIZE = 100

# --- Environment Setup ---
print("🚀 Starting environment setup...")

# 1. Install Python Dependencies
print("\n📦 Installing Python packages...")
packages = [
    "langchain",
    "langchain_community",
    "langchain-google-vertexai",
    "google-cloud-aiplatform",
    "pypdf",
    "tqdm",
    "reportlab",
    "chromadb" # Added ChromaDB client
]
subprocess.run(f"pip install --upgrade -q {' '.join(packages)}", shell=True, check=True)

# 2. Authenticate to Google Cloud (no changes)
print(f"\n🔐 Setting Google Cloud project to: {PROJECT_ID}...")
os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
print("✅ Authentication configured (ensure you've run 'gcloud auth application-default login').")

# 3. Create local directories (no changes)
print("\n📁 Creating local directories...")
KNOWLEDGE_BASE_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"\n✅ Environment is ready! Please add your PDF files to the '{KNOWLEDGE_BASE_DIR}' folder and proceed.")

🚀 Starting environment setup...

📦 Installing Python packages...

🔐 Setting Google Cloud project to: securemed-chat...
✅ Authentication configured (ensure you've run 'gcloud auth application-default login').

📁 Creating local directories...

✅ Environment is ready! Please add your PDF files to the 'knowledge_base' folder and proceed.


In [4]:
# ==============================================================================
# Step 2: Process PDFs & Create/Load Vector Store (ChromaDB Version)
# ==============================================================================
# This step now connects to your remote ChromaDB server.
# If the collection exists, it uses it. Otherwise, it builds a new one in batches.

import chromadb
from langchain_google_vertexai import VertexAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from tqdm import tqdm

# Initialize embeddings and ChromaDB client
embeddings = VertexAIEmbeddings(model_name=EMBEDDING_MODEL, project=PROJECT_ID)
vector_store = None

print(f"🔌 Attempting to connect to ChromaDB at {CHROMA_HOST}:{CHROMA_PORT}...")
try:
    chroma_client = chromadb.HttpClient(host=CHROMA_HOST, port=CHROMA_PORT)
    # Check if the collection already exists
    collections = [c.name for c in chroma_client.list_collections()]
    
    if COLLECTION_NAME in collections:
        print(f"✅ Found existing collection '{COLLECTION_NAME}'. Loading it.")
        vector_store = Chroma(
            client=chroma_client,
            collection_name=COLLECTION_NAME,
            embedding_function=embeddings
        )
        print("👍 Vector store loaded successfully. Skipping PDF processing.")
    else:
        # If no collection is found, proceed with the one-time build process
        print(f"⚠️ No collection named '{COLLECTION_NAME}' found. Building a new one...")

        # 1. Load and Chunk Documents
        print(f"\n📂 Loading documents from {KNOWLEDGE_BASE_DIR}...")
        pdf_files = list(KNOWLEDGE_BASE_DIR.glob("*.pdf"))
        
        if not pdf_files:
            raise FileNotFoundError(f"CRITICAL ERROR: No PDF files found in {KNOWLEDGE_BASE_DIR} to build the vector store.")
        
        print(f"Found {len(pdf_files)} PDF files to process.")
        all_chunks = []
        text_splitter = RecursiveCharacterTextSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)

        for pdf_file in tqdm(pdf_files, desc="Processing PDFs"):
            try:
                loader = PyPDFLoader(str(pdf_file))
                docs = loader.load_and_split(text_splitter=text_splitter)
                all_chunks.extend(docs)
            except Exception as e:
                print(f"Error processing {pdf_file.name}: {e}")

        print(f"\n✅ Successfully created {len(all_chunks)} text chunks.")

        # 2. Create Vector Store in ChromaDB with Batching
        if all_chunks:
            print(f"🧠 Creating new collection '{COLLECTION_NAME}' in batches...")
            
            # Create the store with the first batch to initialize the collection
            vector_store = Chroma.from_documents(
                documents=all_chunks[:BATCH_SIZE],
                embedding=embeddings,
                collection_name=COLLECTION_NAME,
                client=chroma_client
            )
            
            # Add the remaining documents in batches
            remaining_docs = all_chunks[BATCH_SIZE:]
            for i in tqdm(range(0, len(remaining_docs), BATCH_SIZE), desc="Adding document batches to ChromaDB"):
                batch = remaining_docs[i:i + BATCH_SIZE]
                if batch:
                    vector_store.add_documents(batch)
            
            print(f"\n✅ New vector store created and populated in ChromaDB.")
        else:
            print("❌ Cannot create vector store: No document chunks were generated.")

except Exception as e:
    print(f"❌ FAILED TO CONNECT OR BUILD VECTOR STORE: {e}")
    print("   Please check if your ChromaDB server is running and accessible at the specified host/port.")
    print("   Ensure your firewall rule allows connections from your current IP address.")

/home/cazzi/code/caazzi/securemed_chat/.direnv/python-3.10.6/lib/python3.10/site-packages/vertexai/_model_garden/_model_garden_models.py:278: UserWarning: This feature is deprecated as of June 24, 2025 and will be removed on June 24, 2026. For details, see https://cloud.google.com/vertex-ai/generative-ai/docs/deprecations/genai-vertexai-sdk.
  warning_logs.show_deprecation_warning()


🔌 Attempting to connect to ChromaDB at 34.151.247.35:8000...
⚠️ No collection named 'securemed_docs' found. Building a new one...

📂 Loading documents from knowledge_base...
Found 29 PDF files to process.


Processing PDFs: 100%|█████████████████████████████████████████████████████████████████████████████████████████████| 29/29 [36:52<00:00, 76.30s/it]



✅ Successfully created 50982 text chunks.
🧠 Creating new collection 'securemed_docs' in batches...


Adding document batches to ChromaDB: 100%|███████████████████████████████████████████████████████████████████████| 509/509 [55:47<00:00,  6.58s/it]


✅ New vector store created and populated in ChromaDB.


In [ ]:
# ==============================================================================
# Step 3: Build the RAG Chain and LLM
# ==============================================================================
from langchain_google_vertexai import ChatVertexAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# --- Prerequisite Check ---
if 'vector_store' not in locals() or vector_store is None:
    print("❌ Cannot proceed: Vector store is not available. Please run Step 2 first.")
else:
    # 1. Initialize the Gemini Language Model via Vertex AI
    # We use a low temperature for more consistent, factual question generation
    llm = ChatVertexAI(model_name=LLM_MODEL, temperature=0.2, project=PROJECT_ID)

    # 2. Initialize the Retriever from our vector store
    # This component fetches the top 4 most relevant document chunks for a given query.
    retriever = vector_store.as_retriever(search_kwargs={'k': 4})

    # 3. Define the Prompt Template for the first set of questions
    # This instructs the LLM on how to use the retrieved context to generate questions.
    RAG_PROMPT_TEMPLATE = """
System: You are a helpful health assistant. Your goal is to help a patient structure their relevant medical history based on their chief complaint.

CONTEXT:
{context}

QUESTION:
Based ONLY on the clinical anamnesis information in the CONTEXT above, generate a numbered list of exactly 5 essential questions to get more information about the patient's medical history for this chief complaint: '{chief_complaint}'.

Frame the questions from the patient's perspective. For example, instead of "Ask about onset," the question should be "When did this symptom start?". Do not add any conversational text before or after the list of questions.
"""
    rag_prompt = ChatPromptTemplate.from_template(RAG_PROMPT_TEMPLATE)

    # 4. Create the RAG Chain
    # This chain links all the components together.
    rag_chain = (
        {"context": retriever, "chief_complaint": RunnablePassthrough()}
        | rag_prompt
        | llm
        | StrOutputParser()
    )

    print("✅ RAG chain and LLM are ready for the chat.")

In [ ]:
# ==============================================================================
# Step 4: Run the Interactive SecureMed Chat (Fully Updated)
# ==============================================================================
import json
import re
from datetime import datetime
from reportlab.pdfgen import canvas
from reportlab.lib.pagesizes import letter
from reportlab.lib.units import inch
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.platypus import Paragraph

# --- Prerequisite Check ---
# Ensure the necessary components from previous steps are available.
if 'rag_chain' not in locals() or 'llm' not in locals() or vector_store is None:
    print("❌ Cannot proceed: Please run all previous steps (1-3) to build the necessary components.")
else:
    try:
        # --- 1. Start message ---
        print("✅ SecureMed Chat is ready.")
        print("*"*60)
        print("Objective: To help you prepare for your doctor's visit by structuring your symptoms.")
        print("Limitation: I am not a doctor and this is not medical advice. This tool is for informational purposes only. I don't store any information about you.")
        print("*"*60)

        # --- 2. Get user's initial info ---
        chief_complaint_raw = input("\n🩺 To begin, please describe your chief complaint (e.g., 'Persistent dry cough'): ")
        age = input("👤 Please enter your age: ")
        gender = input("👤 Please enter your gender: ")

        # Combine inputs for better contextual understanding by the LLM
        chief_complaint_context = f"A {age}-year-old {gender} presenting with: {chief_complaint_raw}"

        if chief_complaint_raw:
            # --- 3. Create the first question list using RAG ---
            print("\n🧠 Thinking... Generating relevant questions based on your complaint...")
            generated_questions_text_1 = rag_chain.invoke(chief_complaint_context)

            print("\n📝 Based on your symptom, please answer the following questions. Be as detailed as you can.")
            print("-" * 20)
            questions_1 = re.findall(r"^\s*[-*]?\s*\d*\.?\s*(.*)", generated_questions_text_1, re.MULTILINE)
            if questions_1:
                for q in questions_1:
                    if q.strip(): print(f"- {q.strip()}")
            else:
                print(generated_questions_text_1)
            print("-" * 20)

            # --- 4. Get the user's first answers ---
            patient_answers_1 = input("\nYour detailed answer: ")

            if patient_answers_1:
                # --- 5. Generate a second, deeper set of questions ---
                print("\n🧠 Thinking a bit deeper... Generating follow-up questions for your medical history...")

                DEEP_DIVE_PROMPT_TEMPLATE = """
                System: You are a medical assistant. Based on the patient's chief complaint and their initial answers, generate 5 follow-up questions to get a deeper understanding of their relevant medical history (like past surgeries, chronic conditions, family history, and medications related to the complaint).

                PATIENT PROFILE: {chief_complaint}
                PATIENT'S INITIAL ANSWERS: {initial_answers}

                QUESTION:
                Generate a numbered list of exactly 5 essential follow-up questions to explore the patient's deeper medical history. Frame questions from the patient's perspective. Do not add any conversational text.
                """
                deep_dive_prompt = ChatPromptTemplate.from_template(DEEP_DIVE_PROMPT_TEMPLATE)
                deep_dive_chain = deep_dive_prompt | llm | StrOutputParser()
                generated_questions_text_2 = deep_dive_chain.invoke({
                    "chief_complaint": chief_complaint_context,
                    "initial_answers": patient_answers_1
                })

                print("\n📝 To build a more complete picture, please answer these follow-up questions.")
                print("-" * 20)
                questions_2 = re.findall(r"^\s*[-*]?\s*\d*\.?\s*(.*)", generated_questions_text_2, re.MULTILINE)
                if questions_2:
                    for q in questions_2:
                        if q.strip(): print(f"- {q.strip()}")
                else:
                    print(generated_questions_text_2)
                print("-" * 20)
                print("Also, send me any exams that you have related to this complaint and that you want to add at your medical summary.")

                # --- 6. Get the user's second answers ---
                patient_answers_2 = input("\nYour answers to the follow-up questions: ")

                # --- 7. Summarize all information and generate PDF ---
                print("\n📄 Thank you. Summarizing all your answers into a structured format...")
                full_patient_text = f"Initial Answers: {patient_answers_1}\n\nFollow-up History Answers: {patient_answers_2}"

                summarization_prompt = f"""
                Analyze the following patient's text. The patient is a {age}-year-old {gender}.
                Extract the key medical information and structure it as a clean JSON object.
                The JSON should have keys like "chief_complaint", "onset", "character", "associated_symptoms", "past_medical_history", "family_history", and "medications".
                If a key's information is not present, use "N/A".

                Patient's full text: "{full_patient_text}"
                """
                structured_summary_str = llm.invoke(summarization_prompt).content

                try:
                    json_str_cleaned = structured_summary_str.strip().replace("```json", "").replace("```", "")
                    structured_data = json.loads(json_str_cleaned)
                    structured_data['chief_complaint'] = chief_complaint_raw

                    def generate_pdf_report(data):
                        # Get the current date and time to create a unique filename
                        timestamp = datetime.now().strftime("%Y-%m-%d_%H%M%S")
                        file_name = f"Medical_Summary_Report_{timestamp}.pdf"
                        
                        # Saves the PDF to your reports folder with the new unique name
                        pdf_path = str(REPORTS_DIR / file_name)
                        c = canvas.Canvas(pdf_path, pagesize=letter)
                        width, height = letter
                        styles = getSampleStyleSheet()

                        c.setFont("Helvetica", 9)
                        c.drawRightString(width - inch, height - 0.9*inch, f"Generated on: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

                        c.setFont("Helvetica-Bold", 16)
                        c.drawString(inch, height - inch, "Pre-Visit Medical Summary")
                        c.line(inch, height - 1.1*inch, width - inch, height - 1.1*inch)

                        c.setFont("Helvetica-Bold", 12)
                        c.drawString(inch, height - 1.5*inch, "Chief Complaint:")
                        c.setFont("Helvetica", 11)
                        c.drawString(inch * 2.5, height - 1.5*inch, data.get("chief_complaint", "N/A"))

                        c.setFont("Helvetica-Bold", 12)
                        c.drawString(inch, height - 2.0*inch, "History of Present Illness & Past History:")
                        style_body = styles['BodyText']
                        style_body.fontName = 'Helvetica'
                        style_body.fontSize = 11
                        style_body.leading = 14
                        
                        history_text = f"""
                        <b>Onset:</b> {data.get('onset', 'N/A')}<br/>
                        <b>Character:</b> {data.get('character', 'N/A')}<br/>
                        <b>Associated Symptoms:</b> {data.get('associated_symptoms', 'N/A')}<br/>
                        <b>Past Medical History:</b> {data.get('past_medical_history', 'N/A')}<br/>
                        <b>Family History:</b> {data.get('family_history', 'N/A')}<br/>
                        <b>Medications:</b> {data.get('medications', 'N/A')}
                        """
                        p = Paragraph(history_text, style_body)
                        p.wrapOn(c, width - 2*inch, height)
                        p.drawOn(c, inch, height - 4.2*inch)

                        c.save()
                        return pdf_path

                    print("\n💾 Generating PDF report...")
                    pdf_file_path = generate_pdf_report(structured_data)
                    print(f"✅ PDF report successfully saved to your local directory: {pdf_file_path}")

                except json.JSONDecodeError:
                    print("\n❌ Error: Could not parse the summary from the language model. Please try again.")
                    print("--- Raw LLM Output ---")
                    print(structured_summary_str)

    except KeyboardInterrupt:
        print("\n\nProcess cancelled by user. Exiting.")